# Sales Prediction & Inventory Optimization

## Approach:
1. Train Holt-Winters (Triple Exponential Smoothing) models on 2023-2024 data (24 months)
2. Evaluate predictions against actual 2025 data (12 months)
3. Calculate optimal inventory levels based on predicted demand and safety stock
4. Compare model performance with the Financial Plan
5. Estimate additional revenue from avoided stockouts
6. Generate 2026 predictions using full 2023-2025 data (36 months)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy import stats
import warnings

warnings.filterwarnings("ignore")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

## 1. Data Loading

In [ ]:
# Load all CSV files
DATASET_DIR = '.'

sales_train = pd.read_csv(os.path.join(DATASET_DIR, 'sales_train_2023_2024.csv'))
sales_test = pd.read_csv(os.path.join(DATASET_DIR, 'sales_test_2025.csv'))
products = pd.read_csv(os.path.join(DATASET_DIR, 'products_parameters.csv'))
financial_plan = pd.read_csv(os.path.join(DATASET_DIR, 'financial_plan.csv'))

print(f"Sales Train shape: {sales_train.shape}")
print(f"Sales Test shape: {sales_test.shape}")
print(f"Products shape: {products.shape}")
print(f"Financial Plan shape: {financial_plan.shape}")

## 2. Data Preparation

In [ ]:
# Identify column groups
id_cols = ['ProductID', 'Product_name', 'Type']
train_cols = [c for c in sales_train.columns if c not in id_cols]
test_cols = [c for c in sales_test.columns if c not in id_cols]

print(f"Training columns ({len(train_cols)}): {train_cols[0]} - {train_cols[-1]}")
print(f"Test columns ({len(test_cols)}): {test_cols[0]} - {test_cols[-1]}")

# Split by type
df_sales_train = sales_train[sales_train['Type'] == 'Sales'].copy().reset_index(drop=True)
df_stock_train = sales_train[sales_train['Type'] == 'Inventory_level'].copy().reset_index(drop=True)
df_sales_test = sales_test[sales_test['Type'] == 'Sales'].copy().reset_index(drop=True)
df_stock_test = sales_test[sales_test['Type'] == 'Inventory_level'].copy().reset_index(drop=True)

product_ids = df_sales_train['ProductID'].unique()
print(f"\nNumber of products: {len(product_ids)}")
print(f"Sales train rows: {len(df_sales_train)}")
print(f"Stock train rows: {len(df_stock_train)}")

## 3. Model Training - Holt-Winters (Triple Exponential Smoothing)

We use Holt-Winters because:
- Good handling of seasonality (12-month cycle)
- Flexible trend fitting
- Proven effectiveness for short-horizon time series forecasting

In [ ]:
def train_predict_holt_winters(series_train, steps=12, seasonal_periods=12):
    """Train Holt-Winters model and return forecast."""
    try:
        model = ExponentialSmoothing(
            series_train,
            trend='add',
            seasonal='add',
            seasonal_periods=seasonal_periods
        ).fit(optimized=True)
        forecast = model.forecast(steps)
        return np.maximum(forecast.values, 0)
    except Exception:
        # Fallback - seasonal naive (repeat last year)
        if len(series_train) >= seasonal_periods:
            last_season = series_train.values[-seasonal_periods:]
            forecast = np.tile(
                last_season, (steps // seasonal_periods) + 1
            )[:steps]
        else:
            forecast = np.full(steps, series_train.mean())
        return np.maximum(forecast, 0)

In [ ]:
# Train model for each product (on 2023-2024, predict 2025)
predictions = {}
actuals = {}

for pid in product_ids:
    train_series = pd.Series(
        df_sales_train[df_sales_train['ProductID'] == pid][train_cols].values.flatten(),
        dtype=float
    )
    test_series = df_sales_test[
        df_sales_test['ProductID'] == pid
    ][test_cols].values.flatten()

    forecast = train_predict_holt_winters(train_series, steps=12)
    predictions[pid] = forecast
    actuals[pid] = test_series

print(f"Trained models for {len(predictions)} products")

## 4. Model Evaluation (2025 Predictions vs Actual)

In [ ]:
# Evaluate predictions
all_preds = []
all_actuals = []
metrics_per_product = []

for pid in product_ids:
    pred = predictions[pid]
    actual = actuals[pid]
    all_preds.extend(pred)
    all_actuals.extend(actual)

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(
        np.abs((actual - pred) / np.maximum(actual, 1))
    ) * 100

    metrics_per_product.append({
        'ProductID': pid,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape
    })

df_metrics = pd.DataFrame(metrics_per_product)

print("=== Sales Prediction Metrics (Aggregate) ===")
print(f"MAE:  {mean_absolute_error(all_actuals, all_preds):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(all_actuals, all_preds)):.2f}")
print(f"R\u00b2:   {r2_score(all_actuals, all_preds):.4f}")
print(f"\nMean MAPE per product: {df_metrics['MAPE'].mean():.2f}%")
print(f"Median MAPE: {df_metrics['MAPE'].median():.2f}%")

In [ ]:
# Visualize predictions vs actual for sample products
sample_products = [product_ids[0], product_ids[9], product_ids[49],
                   product_ids[99], product_ids[min(199, len(product_ids) - 1)]]

fig, axes = plt.subplots(len(sample_products), 1,
                         figsize=(14, 3 * len(sample_products)))

for i, pid in enumerate(sample_products):
    train_data = df_sales_train[
        df_sales_train['ProductID'] == pid
    ][train_cols].values.flatten()
    test_data = actuals[pid]
    pred_data = predictions[pid]

    train_idx = pd.date_range('2023-01', periods=len(train_cols), freq='MS')
    test_idx = pd.date_range('2025-01', periods=12, freq='MS')

    axes[i].plot(train_idx, train_data, label='Training', color='blue')
    axes[i].plot(test_idx, test_data, label='Actual 2025',
                 color='green', marker='o', markersize=4)
    axes[i].plot(test_idx, pred_data, label='Prediction',
                 color='red', linestyle='--', marker='x', markersize=4)
    axes[i].set_title(
        f"{pid} - MAE: {mean_absolute_error(test_data, pred_data):.1f}"
    )
    axes[i].legend(loc='upper right')
    axes[i].axvline(x=pd.Timestamp('2025-01-01'),
                    color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Optimal Inventory Calculation

**Optimal inventory formula:**

$$\text{Optimal}(t) = \text{Forecast}(t) + \text{Safety\_stock\_months} \times \text{Mean\_future\_demand}$$

Safety stock months defines how many months of future demand should be kept in reserve.

In [ ]:
def calculate_optimal_stock(forecast, safety_stock_months):
    """Calculate optimal inventory level based on forecast and safety stock."""
    optimal = np.zeros(len(forecast))
    for t in range(len(forecast)):
        current_demand = forecast[t]
        future_months = int(np.ceil(safety_stock_months))
        future_demand = 0

        for m in range(1, future_months + 1):
            if t + m < len(forecast):
                future_demand += forecast[t + m]
            else:
                future_demand += np.mean(forecast)

        fraction = safety_stock_months - int(safety_stock_months)
        if fraction > 0:
            if t + future_months + 1 < len(forecast):
                future_demand += fraction * forecast[t + future_months + 1]
            else:
                future_demand += fraction * np.mean(forecast)

        optimal[t] = current_demand + future_demand
    return np.ceil(optimal).astype(int)

In [ ]:
# Calculate optimal inventory for each product
optimal_stocks = {}
actual_stocks = {}

for pid in product_ids:
    params = products[products['ProductID'] == pid].iloc[0]
    safety_months = params['Safety_stock_months']

    forecast = predictions[pid]
    optimal = calculate_optimal_stock(forecast, safety_months)
    optimal_stocks[pid] = optimal

    actual_stock = df_stock_test[
        df_stock_test['ProductID'] == pid
    ][test_cols].values.flatten()
    actual_stocks[pid] = actual_stock

print("Optimal inventory calculated for all products")

In [ ]:
# Compare optimal vs actual inventory
results = []

for pid in product_ids:
    params = products[products['ProductID'] == pid].iloc[0]
    storage_cost = params['Storage_cost_PLN_per_unit_month']

    actual_s = actuals[pid]
    actual_st = actual_stocks[pid]
    optimal_st = optimal_stocks[pid]

    cost_actual = np.sum(actual_st * storage_cost)
    cost_optimal = np.sum(optimal_st * storage_cost)

    surplus_actual = np.sum(actual_st - actual_s)
    surplus_optimal = np.sum(optimal_st - actual_s)

    stockout_actual = np.sum(np.maximum(actual_s - actual_st, 0))
    stockout_optimal = np.sum(np.maximum(actual_s - optimal_st, 0))

    results.append({
        'ProductID': pid,
        'cost_actual_PLN': cost_actual,
        'cost_optimal_PLN': cost_optimal,
        'savings_PLN': cost_actual - cost_optimal,
        'surplus_actual_units': surplus_actual,
        'surplus_optimal_units': surplus_optimal,
        'stockout_actual_units': stockout_actual,
        'stockout_optimal_units': stockout_optimal
    })

df_results = pd.DataFrame(results)

print("=== OPTIMIZATION SUMMARY (Year 2025) ===")
print(f"\nTotal storage cost - actual:  {df_results['cost_actual_PLN'].sum():>12,.2f} PLN")
print(f"Total storage cost - optimal: {df_results['cost_optimal_PLN'].sum():>12,.2f} PLN")
print(f"Potential savings:            {df_results['savings_PLN'].sum():>12,.2f} PLN")
print(f"Savings percentage: {df_results['savings_PLN'].sum() / df_results['cost_actual_PLN'].sum() * 100:.1f}%")
print(f"\nTotal surplus actual:  {df_results['surplus_actual_units'].sum():,.0f} units")
print(f"Total surplus optimal: {df_results['surplus_optimal_units'].sum():,.0f} units")
print(f"\nStockouts - actual:  {df_results['stockout_actual_units'].sum():,.0f} units")
print(f"Stockouts - optimal: {df_results['stockout_optimal_units'].sum():,.0f} units")

In [ ]:
# Visualize optimal vs actual inventory for sample products
fig, axes = plt.subplots(len(sample_products), 1,
                         figsize=(14, 3.5 * len(sample_products)))
test_idx = pd.date_range('2025-01', periods=12, freq='MS')

for i, pid in enumerate(sample_products):
    axes[i].bar(test_idx, actuals[pid], alpha=0.4,
               label='Sales (actual)', color='green', width=20)
    axes[i].plot(test_idx, actual_stocks[pid],
                label='Actual inventory', color='blue',
                marker='o', markersize=4)
    axes[i].plot(test_idx, optimal_stocks[pid],
                label='Optimal inventory (model)', color='red',
                marker='x', markersize=6, linestyle='--')
    axes[i].set_title(f"{pid}")
    axes[i].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 6. Comparison with Financial Plan

In [ ]:
# Compare model predictions vs financial plan vs actual
comparison = []

for pid in product_ids:
    plan_row = financial_plan[financial_plan['ProductID'] == pid]
    params = products[products['ProductID'] == pid].iloc[0]

    if plan_row.empty:
        continue

    plan_row = plan_row.iloc[0]

    actual_sales_2025 = actuals[pid]
    actual_total_2025 = actual_sales_2025.sum()
    actual_stock_2025 = actual_stocks[pid]

    model_pred_total = predictions[pid].sum()
    plan_total = plan_row['Plan_2025_units']

    storage_cost = params['Storage_cost_PLN_per_unit_month']
    price = params['Price_PLN_per_unit']

    cost_actual = actual_stock_2025.sum() * storage_cost
    cost_model = optimal_stocks[pid].sum() * storage_cost

    # Plan storage cost estimate (uniform monthly stock)
    plan_monthly_sales = plan_total / 12
    plan_stock_estimate = plan_monthly_sales * (1 + params['Safety_stock_months'])
    cost_plan = plan_stock_estimate * 12 * storage_cost

    error_model = abs(model_pred_total - actual_total_2025)
    error_plan = abs(plan_total - actual_total_2025)

    comparison.append({
        'ProductID': pid,
        'actual_sales_2025': actual_total_2025,
        'model_prediction': round(model_pred_total),
        'financial_plan': plan_total,
        'error_model_units': round(error_model),
        'error_plan_units': round(error_plan),
        'storage_cost_actual_PLN': round(cost_actual, 2),
        'storage_cost_model_PLN': round(cost_model, 2),
        'storage_cost_plan_PLN': round(cost_plan, 2),
        'revenue_actual_PLN': round(actual_total_2025 * price, 2),
        'revenue_model_PLN': round(model_pred_total * price, 2),
        'revenue_plan_PLN': round(plan_total * price, 2)
    })

df_comparison = pd.DataFrame(comparison)
df_comparison.head(10)

In [ ]:
# Summary comparison
print("=" * 70)
print("COMPARISON: MODEL vs FINANCIAL PLAN vs ACTUAL (2025)")
print("=" * 70)

print("\n--- SALES PREDICTION ACCURACY ---")
print(f"Mean model error (MAE annual):     {df_comparison['error_model_units'].mean():.1f} units")
print(f"Mean plan error (MAE annual):      {df_comparison['error_plan_units'].mean():.1f} units")

model_mape = (
    df_comparison['error_model_units']
    / df_comparison['actual_sales_2025'].replace(0, np.nan) * 100
)
plan_mape = (
    df_comparison['error_plan_units']
    / df_comparison['actual_sales_2025'].replace(0, np.nan) * 100
)
print(f"Mean MAPE model:                   {model_mape.mean():.1f}%")
print(f"Mean MAPE plan:                    {plan_mape.mean():.1f}%")

print("\n--- STORAGE COSTS (Year 2025) ---")
print(f"Cost actual:  {df_comparison['storage_cost_actual_PLN'].sum():>12,.2f} PLN")
print(f"Cost model:   {df_comparison['storage_cost_model_PLN'].sum():>12,.2f} PLN")
print(f"Cost plan:    {df_comparison['storage_cost_plan_PLN'].sum():>12,.2f} PLN")

savings_vs_actual = (
    df_comparison['storage_cost_actual_PLN'].sum()
    - df_comparison['storage_cost_model_PLN'].sum()
)
savings_vs_plan = (
    df_comparison['storage_cost_plan_PLN'].sum()
    - df_comparison['storage_cost_model_PLN'].sum()
)
pct_savings = (
    savings_vs_actual / df_comparison['storage_cost_actual_PLN'].sum() * 100
)

print(f"\nModel savings vs actual: {savings_vs_actual:>10,.2f} PLN ({pct_savings:.1f}%)")
print(f"Model savings vs plan:   {savings_vs_plan:>10,.2f} PLN")

print("\n--- REVENUE ---")
print(f"Revenue actual: {df_comparison['revenue_actual_PLN'].sum():>12,.2f} PLN")
print(f"Revenue model:  {df_comparison['revenue_model_PLN'].sum():>12,.2f} PLN")
print(f"Revenue plan:   {df_comparison['revenue_plan_PLN'].sum():>12,.2f} PLN")

In [ ]:
# Which solution is better per product?
df_comparison['model_better_sales'] = (
    df_comparison['error_model_units'] < df_comparison['error_plan_units']
)
df_comparison['model_cheaper_storage'] = (
    df_comparison['storage_cost_model_PLN']
    < df_comparison['storage_cost_actual_PLN']
)

print("--- Model vs Plan (sales prediction accuracy) ---")
print(f"Model wins: {df_comparison['model_better_sales'].sum()} / {len(df_comparison)} products")
print(f"Plan wins:  {(~df_comparison['model_better_sales']).sum()} / {len(df_comparison)} products")

print(f"\n--- Model cheaper than actual inventory ---")
print(f"Model cheaper: {df_comparison['model_cheaper_storage'].sum()} / {len(df_comparison)} products")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Prediction error: model vs plan
axes[0, 0].scatter(
    df_comparison['error_plan_units'],
    df_comparison['error_model_units'], alpha=0.5, s=20
)
max_val = max(
    df_comparison['error_plan_units'].max(),
    df_comparison['error_model_units'].max()
)
axes[0, 0].plot([0, max_val], [0, max_val], 'r--', label='Model = Plan')
axes[0, 0].set_xlabel('Plan error [units]')
axes[0, 0].set_ylabel('Model error [units]')
axes[0, 0].set_title('Accuracy: Model vs Plan (below line = model better)')
axes[0, 0].legend()

# 2. Storage costs comparison
categories = ['Actual', 'Model', 'Plan']
values = [
    df_comparison['storage_cost_actual_PLN'].sum(),
    df_comparison['storage_cost_model_PLN'].sum(),
    df_comparison['storage_cost_plan_PLN'].sum()
]
colors = ['steelblue', 'green', 'orange']
axes[0, 1].bar(categories, values, color=colors, edgecolor='black')
axes[0, 1].set_title('Total Storage Costs 2025')
axes[0, 1].set_ylabel('PLN')
for i, v in enumerate(values):
    axes[0, 1].text(i, v + max(values) * 0.01, f"{v:,.0f}",
                    ha='center', fontsize=9)

# 3. MAPE distribution
axes[1, 0].hist(model_mape.dropna(), bins=25, alpha=0.6,
               label='Model', color='green', edgecolor='black')
axes[1, 0].hist(plan_mape.dropna(), bins=25, alpha=0.6,
               label='Plan', color='orange', edgecolor='black')
axes[1, 0].set_title('MAPE Distribution - Model vs Plan')
axes[1, 0].set_xlabel('MAPE [%]')
axes[1, 0].legend()

# 4. Savings per product
savings = (
    df_comparison['storage_cost_actual_PLN']
    - df_comparison['storage_cost_model_PLN']
).sort_values(ascending=False)
axes[1, 1].bar(
    range(len(savings)), savings.values,
    color=['green' if x > 0 else 'red' for x in savings.values],
    width=1
)
axes[1, 1].set_title('Model Savings vs Actual (per product)')
axes[1, 1].set_xlabel('Products (sorted)')
axes[1, 1].set_ylabel('Savings [PLN]')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 7. Additional Revenue from Avoided Stockouts

We estimate profit from additional sales in months where actual inventory was 0 (stockout) but our model predicted demand and maintained stock. Profit = quantity × price × 20% margin.

In [ ]:
MARGIN = 0.20

additional_revenue = []

for pid in product_ids:
    actual_stock_2025 = actual_stocks[pid]
    forecast = predictions[pid]
    params = products[products['ProductID'] == pid].iloc[0]
    price = params['Price_PLN_per_unit']

    profit_product = 0
    stockout_months = 0

    for t in range(len(test_cols)):
        if actual_stock_2025[t] == 0 and forecast[t] > 0:
            extra_sales = forecast[t]
            profit = extra_sales * price * MARGIN
            profit_product += profit
            stockout_months += 1

    additional_revenue.append({
        'ProductID': pid,
        'stockout_months': stockout_months,
        'extra_sales_units': round(sum(
            forecast[t] for t in range(len(test_cols))
            if actual_stock_2025[t] == 0 and forecast[t] > 0
        )),
        'extra_profit_PLN': round(profit_product, 2),
        'price_PLN': price
    })

df_extra = pd.DataFrame(additional_revenue)

print("=" * 70)
print("ADDITIONAL REVENUE FROM AVOIDED STOCKOUTS")
print("=" * 70)
print(f"\nProducts with stockouts in 2025:  {(df_extra['stockout_months'] > 0).sum()} / {len(df_extra)}")
print(f"Total stockout-months:            {df_extra['stockout_months'].sum()}")
print(f"Total extra sales:                {df_extra['extra_sales_units'].sum():,.0f} units")
print(f"Total extra profit (20% margin):  {df_extra['extra_profit_PLN'].sum():,.2f} PLN")

print("\nTop 10 products - highest extra profit:")
top10_profit = df_extra.nlargest(10, 'extra_profit_PLN')[
    ['ProductID', 'stockout_months', 'extra_sales_units', 'extra_profit_PLN']
]
print(top10_profit.to_string(index=False))

In [ ]:
# Confidence intervals (Bootstrap + prediction error)

# Method 1: Based on prediction error distribution
pct_errors = []
for pid in product_ids:
    actual_sales_2025 = actuals[pid]
    forecast = predictions[pid]
    for t in range(len(test_cols)):
        if actual_sales_2025[t] > 0:
            err = (forecast[t] - actual_sales_2025[t]) / actual_sales_2025[t]
            pct_errors.append(err)

pct_errors = np.array(pct_errors)
mean_error = pct_errors.mean()
std_error = pct_errors.std()

total_extra_profit = df_extra['extra_profit_PLN'].sum()
profit_adjusted = total_extra_profit / (1 + mean_error)

ci_lower_pct = np.percentile(pct_errors, 2.5)
ci_upper_pct = np.percentile(pct_errors, 97.5)
profit_ci_lower = total_extra_profit / (1 + ci_upper_pct)
profit_ci_upper = total_extra_profit / (1 + ci_lower_pct)

# Method 2: Bootstrap
n_bootstrap = 10_000
profits_per_product = df_extra['extra_profit_PLN'].values
bootstrap_profits = np.array([
    np.random.choice(profits_per_product, size=len(profits_per_product),
                     replace=True).sum()
    for _ in range(n_bootstrap)
])

ci_boot_lower = np.percentile(bootstrap_profits, 2.5)
ci_boot_upper = np.percentile(bootstrap_profits, 97.5)

print("=" * 70)
print("CONFIDENCE INTERVALS - EXTRA PROFIT FROM AVOIDED STOCKOUTS")
print("=" * 70)

print(f"\n--- Method 1: Prediction error distribution ---")
print(f"Mean prediction error:  {mean_error * 100:.1f}%")
print(f"Std prediction error:   {std_error * 100:.1f}%")
print(f"Point estimate:         {total_extra_profit:>12,.2f} PLN")
print(f"Bias-corrected:         {profit_adjusted:>12,.2f} PLN")
print(f"95% CI:                 [{profit_ci_lower:>10,.2f} , {profit_ci_upper:>10,.2f}] PLN")

print(f"\n--- Method 2: Bootstrap ({n_bootstrap:,} iterations) ---")
print(f"Mean bootstrap:         {bootstrap_profits.mean():>12,.2f} PLN")
print(f"95% CI:                 [{ci_boot_lower:>10,.2f} , {ci_boot_upper:>10,.2f}] PLN")

In [ ]:
# Confidence intervals - storage cost savings
savings_per_product = (
    df_comparison['storage_cost_actual_PLN']
    - df_comparison['storage_cost_model_PLN']
).values

total_savings = savings_per_product.sum()

bootstrap_savings = np.array([
    np.random.choice(savings_per_product, size=len(savings_per_product),
                     replace=True).sum()
    for _ in range(n_bootstrap)
])

ci_sav_lower = np.percentile(bootstrap_savings, 2.5)
ci_sav_upper = np.percentile(bootstrap_savings, 97.5)

print("=" * 70)
print("CONFIDENCE INTERVALS - STORAGE COST SAVINGS")
print("=" * 70)
print(f"\nPoint estimate:   {total_savings:>12,.2f} PLN")
print(f"Bootstrap 95% CI: [{ci_sav_lower:>10,.2f} , {ci_sav_upper:>10,.2f}] PLN")

In [ ]:
# Per-product summary
n_products = len(df_comparison)
n_products_stockout = (df_extra['stockout_months'] > 0).sum()

avg_savings = total_savings / n_products
avg_extra_profit = total_extra_profit / n_products
total_benefit = total_savings + total_extra_profit
avg_total_benefit = total_benefit / n_products

# Combined CI (bootstrap)
combined_per_product = savings_per_product + df_extra['extra_profit_PLN'].values
bootstrap_combined = np.array([
    np.random.choice(combined_per_product, size=len(combined_per_product),
                     replace=True).sum()
    for _ in range(n_bootstrap)
])
ci_comb_lower = np.percentile(bootstrap_combined, 2.5)
ci_comb_upper = np.percentile(bootstrap_combined, 97.5)

print("=" * 70)
print("TOTAL BENEFIT SUMMARY (Year 2025)")
print("=" * 70)

print(f"\nTotal products: {n_products}")
print(f"Products with stockouts: {n_products_stockout}")

print(f"\n--- STORAGE COST SAVINGS (per product) ---")
print(f"Average savings:  {avg_savings:>10,.2f} PLN/product/year")
print(f"Median savings:   {np.median(savings_per_product):>10,.2f} PLN/product/year")

print(f"\n--- EXTRA PROFIT FROM AVOIDED STOCKOUTS (per product) ---")
print(f"Average (all):        {avg_extra_profit:>10,.2f} PLN/product/year")
avg_stockout_only = total_extra_profit / max(n_products_stockout, 1)
print(f"Average (w/ stockout): {avg_stockout_only:>10,.2f} PLN/product/year")

print(f"\n--- COMBINED BENEFIT (savings + extra profit) ---")
print(f"Total annual benefit:  {total_benefit:>12,.2f} PLN/year")
print(f"Average per product:   {avg_total_benefit:>10,.2f} PLN/product/year")
print(f"\n95% CI (bootstrap):    [{ci_comb_lower:>10,.2f} , {ci_comb_upper:>10,.2f}] PLN/year")
print(f"Per product 95% CI:    [{ci_comb_lower/n_products:>8,.2f} , {ci_comb_upper/n_products:>8,.2f}] PLN/product/year")

## 8. Predictions for 2026

Retrain Holt-Winters on full 2023-2025 data (36 months) and generate forecasts for 2026.

In [ ]:
# Combine train + test for full history
all_month_cols = train_cols + test_cols

# Merge sales train and test into one dataframe
df_sales_full = df_sales_train[['ProductID', 'Product_name', 'Type'] + train_cols].merge(
    df_sales_test[['ProductID'] + test_cols],
    on='ProductID'
)

print(f"Full sales data: {df_sales_full.shape}")
print(f"Date range: {all_month_cols[0]} - {all_month_cols[-1]} ({len(all_month_cols)} months)")

In [ ]:
# Train on full 2023-2025 data and predict 2026
predictions_2026 = {}
optimal_stocks_2026 = {}

for pid in product_ids:
    full_series = pd.Series(
        df_sales_full[
            df_sales_full['ProductID'] == pid
        ][all_month_cols].values.flatten(),
        dtype=float
    )

    forecast = train_predict_holt_winters(full_series, steps=12)
    predictions_2026[pid] = forecast

    params = products[products['ProductID'] == pid].iloc[0]
    optimal_stocks_2026[pid] = calculate_optimal_stock(
        forecast, params['Safety_stock_months']
    )

print(f"2026 predictions generated for {len(predictions_2026)} products")

In [ ]:
# Create output CSV with 2026 predictions
pred_cols_2026 = [f"2026-{m:02d}" for m in range(1, 13)]

rows_sales = []
rows_stock = []

for pid in product_ids:
    name = df_sales_full[
        df_sales_full['ProductID'] == pid
    ]['Product_name'].iloc[0]

    row_s = {'ProductID': pid, 'Product_name': name, 'Type': 'Sales'}
    for j, col in enumerate(pred_cols_2026):
        row_s[col] = int(round(predictions_2026[pid][j]))
    rows_sales.append(row_s)

    row_st = {'ProductID': pid, 'Product_name': name, 'Type': 'Inventory_level'}
    for j, col in enumerate(pred_cols_2026):
        row_st[col] = int(optimal_stocks_2026[pid][j])
    rows_stock.append(row_st)

df_pred_2026 = pd.DataFrame(rows_sales + rows_stock)
df_pred_2026 = df_pred_2026.sort_values(
    ['ProductID', 'Type']
).reset_index(drop=True)

df_pred_2026.to_csv('predictions_2026.csv', index=False)
print(f"Saved: predictions_2026.csv")
print(f"Shape: {df_pred_2026.shape}")
df_pred_2026.head(10)

In [ ]:
# 2026 prediction summary
df_pred_sales_2026 = df_pred_2026[df_pred_2026['Type'] == 'Sales']
df_pred_stock_2026 = df_pred_2026[df_pred_2026['Type'] == 'Inventory_level']

total_sales_2026 = df_pred_sales_2026[pred_cols_2026].sum().sum()
total_stock_2026 = df_pred_stock_2026[pred_cols_2026].sum().sum()

# Historical comparison
cols_2023 = [c for c in train_cols if c.startswith('2023')]
cols_2024 = [c for c in train_cols if c.startswith('2024')]

total_sales_2023 = df_sales_train[cols_2023].sum().sum()
total_sales_2024 = df_sales_train[cols_2024].sum().sum()
total_sales_2025 = df_sales_test[test_cols].sum().sum()

print("=== 2026 PREDICTION SUMMARY ===")
print(f"\nTotal predicted sales 2026:     {total_sales_2026:,.0f} units")
print(f"Total optimal inventory 2026:   {total_stock_2026:,.0f} units")
print(f"\nAnnual sales comparison:")
print(f"  2023: {total_sales_2023:>10,.0f} units")
print(f"  2024: {total_sales_2024:>10,.0f} units")
print(f"  2025: {total_sales_2025:>10,.0f} units")
print(f"  2026: {total_sales_2026:>10,.0f} units (prediction)")
print(f"\nChange 2026 vs 2025: {(total_sales_2026 / total_sales_2025 - 1) * 100:+.1f}%")

In [ ]:
# Estimated 2026 financials
total_storage_cost_2026 = 0
total_revenue_2026 = 0

for pid in product_ids:
    params = products[products['ProductID'] == pid].iloc[0]
    storage_cost = params['Storage_cost_PLN_per_unit_month']
    price = params['Price_PLN_per_unit']

    stock_sum = optimal_stocks_2026[pid].sum()
    sales_sum = predictions_2026[pid].sum()

    total_storage_cost_2026 += stock_sum * storage_cost
    total_revenue_2026 += sales_sum * price

print(f"\n=== 2026 FINANCIAL ESTIMATE ===")
print(f"Predicted revenue:              {total_revenue_2026:>12,.2f} PLN")
print(f"Optimal storage cost:           {total_storage_cost_2026:>12,.2f} PLN")
print(f"Margin after storage costs:     {total_revenue_2026 - total_storage_cost_2026:>12,.2f} PLN")

In [ ]:
# Visualize 2026 predictions for sample products
fig, axes = plt.subplots(len(sample_products), 1,
                         figsize=(14, 3 * len(sample_products)))

hist_idx = pd.date_range('2023-01', periods=36, freq='MS')
pred_idx = pd.date_range('2026-01', periods=12, freq='MS')

for i, pid in enumerate(sample_products):
    hist_data = df_sales_full[
        df_sales_full['ProductID'] == pid
    ][all_month_cols].values.flatten()
    pred_data = predictions_2026[pid]
    opt_stock = optimal_stocks_2026[pid]

    axes[i].plot(hist_idx, hist_data, label='Historical sales', color='blue')
    axes[i].plot(pred_idx, pred_data, label='Prediction 2026',
                color='red', linestyle='--', marker='x', markersize=5)
    axes[i].plot(pred_idx, opt_stock, label='Optimal inventory',
                color='green', linestyle=':', marker='s', markersize=4)
    axes[i].set_title(f"{pid}")
    axes[i].legend(loc='upper right', fontsize=8)
    axes[i].axvline(x=pd.Timestamp('2026-01-01'),
                    color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly aggregated 2026 predictions
monthly_pred = df_pred_sales_2026[pred_cols_2026].sum()
monthly_stock = df_pred_stock_2026[pred_cols_2026].sum()

fig, ax = plt.subplots(figsize=(12, 5))
x = range(12)
ax.bar(x, monthly_pred.values, alpha=0.6,
       label='Sales forecast', color='steelblue', width=0.4)
ax.bar([i + 0.4 for i in x], monthly_stock.values, alpha=0.6,
       label='Optimal inventory', color='green', width=0.4)
ax.set_xticks([i + 0.2 for i in x])
ax.set_xticklabels(pred_cols_2026, rotation=45)
ax.set_title('2026 Prediction - Monthly Sales and Optimal Inventory')
ax.set_ylabel('Units')
ax.legend()
plt.tight_layout()
plt.show()

print("\nDone! File predictions_2026.csv contains complete forecasts.")

In [ ]:
# Save optimization results
df_results.to_csv('optimization_results_2025.csv', index=False)
df_comparison.to_csv('model_vs_plan_comparison.csv', index=False)
print("Saved: optimization_results_2025.csv")
print("Saved: model_vs_plan_comparison.csv")